# 04 · Logistic regression

Compare default and balanced class weights using the same preprocessing, split and training folds as notebooks 02–03

## Load and split data

In [10]:
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.compose import ColumnTransformer
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

root = Path.cwd()
if not (root / "data").is_dir():
    root = root.parent

df = pd.read_csv(root / "data/raw/online_shoppers_intention.csv").drop_duplicates()
X = df.drop(columns="Revenue")
X["Weekend"] = X["Weekend"].astype(int)
y = df["Revenue"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## Preprocessing

In [11]:
numeric_features = [
    "Administrative", "Administrative_Duration", "Informational",
    "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
    "BounceRates", "ExitRates", "PageValues", "SpecialDay",
]
categorical_features = [
    "Month", "VisitorType", "OperatingSystems", "Browser", "Region", "TrafficType",
]

preprocessor = ColumnTransformer([
    ("numeric", StandardScaler(), numeric_features),
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("weekend", "passthrough", ["Weekend"]),
])

## Training cross-validation

Purchase-class F1 selects the configuration. Precision, recall and accuracy provide context. Report mean and standard deviation over the same five folds.

In [12]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_folds = list(cv.split(X_train, y_train))

scoring = {
    "f1": make_scorer(f1_score, pos_label=True, zero_division=0),
    "precision": make_scorer(precision_score, pos_label=True, zero_division=0),
    "recall": make_scorer(recall_score, pos_label=True, zero_division=0),
    "accuracy": "accuracy",
}


## Compare class weights

Logistic regression provides a simple learned model for this binary outcome. Compare only `class_weight=None` and `balanced`; keep `C=1.0`, the default regularisation strength, and the `lbfgs` solver. Use `max_iter=2000` to allow convergence. Balanced weights are calculated from each training fold. Preprocessing is fitted inside each fold. Use the default prediction threshold; no other settings are tuned. Convergence warnings stop evaluation so incomplete fits cannot silently enter the comparison.

In [13]:
rows = []
with warnings.catch_warnings():
    warnings.simplefilter("error", ConvergenceWarning)
    for class_weight in [None, "balanced"]:
        model = Pipeline([
            ("preprocessing", preprocessor),
            ("model", LogisticRegression(
                class_weight=class_weight, C=1.0, solver="lbfgs", max_iter=2000
            )),
        ])
        scores = cross_validate(
            model, X_train, y_train, cv=cv_folds, scoring=scoring,
            error_score="raise",
        )
        row = {"class_weight": "None" if class_weight is None else class_weight}
        for metric in scoring:
            row[f"{metric}_mean"] = scores[f"test_{metric}"].mean()
            row[f"{metric}_std"] = scores[f"test_{metric}"].std()
        rows.append(row)

results = pd.DataFrame(rows).set_index("class_weight")
display(results.style.format("{:.2%}"))
best_weight = results["f1_mean"].idxmax()
print(f"Selected class_weight: {best_weight}")

,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,accuracy_mean,accuracy_std
class_weight,,,,,,,,
None,49.45%,2.77%,74.15%,3.03%,37.15%,2.75%,88.15%,0.48%
balanced,60.63%,2.44%,50.80%,2.35%,75.23%,2.85%,84.72%,1.10%


Selected class_weight: balanced


## Findings

Balanced class weights give the higher mean purchase-class F1: 60.63% (standard deviation 2.44%), compared with 49.45% (2.77%) for default weights. Select `class_weight="balanced"` for the later model comparison.

Balancing raises purchase recall from 37.15% to 75.23%, while precision falls from 74.15% to 50.80%. It finds more purchases but also predicts more purchases incorrectly. Accuracy falls from 88.15% to 84.72%, showing why accuracy alone is not the selection metric for this imbalanced dataset.

Both configurations improve on the majority-class baseline's zero purchase F1.

## References

- [scikit-learn: logistic regression](https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression) - binary classification and regularisation.
- [scikit-learn: LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) - balanced class weights, regularisation strength, solver and iteration limit.
- [scikit-learn: cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html) - training-only evaluation and preprocessing within each fold.
- [scikit-learn: cross_validate](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_validate.html) - evaluating multiple metrics on the same folds.